# VAR 2026 GraphDeCo Train + Render

Notebook này train GraphDeCo cho một scene, render `test/test_poses.csv`, rồi export archive đem về repo local.

Output contract sau khi giải nén vào local `runs/`:

```text
runs/graphdeco/<scene>/point_cloud/iteration_<N>/point_cloud.ply
runs/graphdeco/<scene>/renders_test/<image_name>
runs/_prepared/graphdeco/<scene>/test/test_poses.csv
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Config

Đổi `SCENE`, `DATA_ROOT` và `REPO_URL` cho đúng Drive/repo của bạn. `DATA_ROOT` là folder chứa các scene, ví dụ `.../private_set1` hoặc `.../public_set`.

In [ ]:
SCENE = 'HCM0254'
DATA_ROOT = '/content/drive/MyDrive/VAI_NVS_DATA/phase1/private_set1'
REPO_URL = 'https://github.com/kain205/Novel-View-Synthesis.git'
BRANCH = 'main'
ITERATIONS = 30000
EXTRA_TRAIN_ARGS = ''
EXPORT_DIR = '/content/drive/MyDrive/var2026_colab_exports'


## Clone Repo + Install Dependencies

Colab không dùng Conda env `var2026`/`graphdeco`; cell này cài trực tiếp vào runtime Colab. Nếu runtime reset thì chạy lại từ đầu.

In [ ]:
%cd /content
!rm -rf Novel-View-Synthesis
!git clone --recursive --branch "$BRANCH" "$REPO_URL" Novel-View-Synthesis
%cd /content/Novel-View-Synthesis
!python -m pip install -U pip setuptools wheel ninja cmake
!python -m pip install typer PyYAML Pillow pandas rich plyfile tqdm opencv-python joblib matplotlib scipy
!python -m pip install methods/graphdeco/submodules/simple-knn --no-build-isolation
!python -m pip install methods/graphdeco/submodules/diff-gaussian-rasterization --no-build-isolation
!python -m pip install methods/graphdeco/submodules/fused-ssim --no-build-isolation


## Prepare Scene

`prepare-graphdeco` dùng COLMAP để undistort scene rồi ghi vào `runs/_prepared/graphdeco/<scene>`. Nếu Colab không có `colmap`, cell này sẽ cài bằng apt.

In [ ]:
!apt-get update -y
!apt-get install -y colmap
SCENE_ROOT = f'{DATA_ROOT}/{SCENE}'
PREPARED_ROOT = f'/content/Novel-View-Synthesis/runs/_prepared/graphdeco/{SCENE}'
RUN_ROOT = f'/content/Novel-View-Synthesis/runs/graphdeco/{SCENE}'
!python -m var2026 prepare-graphdeco --scene "$SCENE_ROOT" --out "$PREPARED_ROOT"


## Train GraphDeCo

Model/weight sẽ nằm trong `runs/graphdeco/<scene>/point_cloud/iteration_<N>/point_cloud.ply`.

In [ ]:
!python methods/graphdeco/train.py \
  -s "$PREPARED_ROOT/train" \
  -m "$RUN_ROOT" \
  --iterations "$ITERATIONS" \
  $EXTRA_TRAIN_ARGS


## Render Test Poses

Render output sẽ nằm trong `runs/graphdeco/<scene>/renders_test/` và phải đúng tên ảnh trong CSV.

In [ ]:
!python var2026/runners/graphdeco_render.py \
  --poses "$SCENE_ROOT/test/test_poses.csv" \
  --run "$RUN_ROOT" \
  --out "$RUN_ROOT/renders_test"


## Export Archive

Archive này giải nén vào local `runs/` bằng `tar -xzf <archive> -C runs`.

In [ ]:
import os
from pathlib import Path

export_dir = Path(EXPORT_DIR)
export_dir.mkdir(parents=True, exist_ok=True)
archive = export_dir / f'{SCENE}_graphdeco_colab_export.tar.gz'

!mkdir -p "runs/_prepared/graphdeco/$SCENE/test"
!cp "$SCENE_ROOT/test/test_poses.csv" "runs/_prepared/graphdeco/$SCENE/test/test_poses.csv"
!tar -czf "$archive" -C runs "graphdeco/$SCENE" "_prepared/graphdeco/$SCENE/test/test_poses.csv"
print(f'Exported: {archive}')
